## CREATE demographics data for customer analysis with o3 modell

In [0]:
from pyspark.sql import functions as F, types as T

In [0]:
csv_path = "/Volumes/main_catalog/online_retail/raw/demographic_df.csv"

demog_df = (spark.read
              .option("header", "true")
              .option("sep",    ",")
              .option("inferSchema", "true")   
              .csv(csv_path))

demog_df = demog_df.withColumn(
    "CustomerID",
    F.col("CustomerID").cast(T.DecimalType(10, 0))
)


(demog_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("main_catalog.online_retail.customers"))

In [0]:
%sql 

SELECT *
FROM main_catalog.online_retail.customers c LIMIT 100;

In [0]:
%sql
SELECT 
      cust.gender, 
      COUNT(DISTINCT trx.CustomerID) AS total_customers
FROM main_catalog.online_retail.sales_silver trx
JOIN main_catalog.online_retail.customers cust 
ON trx.CustomerID = cust.CustomerID
GROUP BY cust.gender